# Stock Inventory Level Forecasting

End-to-end ML pipeline for forecasting daily stock inventory levels across multiple SKUs.

| Step | Description |
|------|-------------|
| 1 | **Configuration** — Set catalog, schema, model and endpoint names |
| 2 | **Synthetic Data** — Generate daily inventory for 5 SKUs with trend, seasonality & replenishment |
| 3 | **EDA** — Visualise time series patterns, distribution and seasonal effects |
| 4 | **Databricks AutoML** — Auto-train & tune forecasting models (10-minute budget) |
| 5 | **Prediction** — Load best model, generate 30-day forward forecasts |
| 6 | **Model Registry** — Register champion model to Unity Catalog |
| 7 | **REST Endpoint** — Deploy model serving endpoint and validate with a live call |

In [0]:
# ── Configuration ───────────────────────────────────────────────────────────
# Update CATALOG / SCHEMA to match your Unity Catalog setup.
CATALOG        = "main"
SCHEMA         = "default"
MODEL_NAME     = f"{CATALOG}.{SCHEMA}.inventory_forecast_model"
ENDPOINT_NAME  = "inventory-forecast-endpoint"
EXPERIMENT_DIR = f"/Users/mufajjul.ali@microsoft.com/experiments/inventory_forecast"

print(f"Model registry target : {MODEL_NAME}")
print(f"Serving endpoint      : {ENDPOINT_NAME}")
print(f"MLflow experiment dir : {EXPERIMENT_DIR}")

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

products   = [f"SKU-{i:03d}" for i in range(1, 6)]
start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 6, 30)
date_range = pd.date_range(start_date, end_date, freq="D")

records = []
for product_id in products:
    base_level    = np.random.randint(800, 1500)
    trend_slope   = np.random.uniform(-0.05, 0.15)
    seasonal_amp  = np.random.uniform(100, 300)
    weekly_phase  = np.random.uniform(0, 2 * np.pi)

    for i, date in enumerate(date_range):
        trend         = trend_slope * i
        weekly        = seasonal_amp * 0.3 * np.sin(2 * np.pi * i / 7 + weekly_phase)
        annual        = seasonal_amp * np.sin(2 * np.pi * i / 365.25)
        noise         = np.random.normal(0, 25)
        replenishment = 450 if (i % 28 == 0) else 0          # monthly restock spike
        inventory_level = max(0.0, base_level + trend + weekly + annual + noise + replenishment)
        records.append({
            "date":            date,
            "product_id":      product_id,
            "inventory_level": round(inventory_level, 2),
        })

inventory_df = pd.DataFrame(records)
print(f"Shape       : {inventory_df.shape}")
print(f"Date range  : {inventory_df['date'].min().date()} -> {inventory_df['date'].max().date()}")
print(f"Products    : {inventory_df['product_id'].unique().tolist()}")
print(f"Avg level   : {inventory_df['inventory_level'].mean():.1f}")
display(inventory_df.head(10))

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

inventory_df["day_of_week"] = inventory_df["date"].dt.dayofweek
inventory_df["month"]       = inventory_df["date"].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Stock Inventory — Exploratory Data Analysis", fontsize=13, fontweight="bold")

# Time series per SKU
for pid in inventory_df["product_id"].unique():
    sub = inventory_df[inventory_df["product_id"] == pid].sort_values("date")
    axes[0, 0].plot(sub["date"], sub["inventory_level"], label=pid, alpha=0.75, linewidth=0.7)
axes[0, 0].set_title("Inventory Level Over Time (by SKU)")
axes[0, 0].set_xlabel("Date"); axes[0, 0].set_ylabel("Inventory Level")
axes[0, 0].legend(fontsize=8)
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45)

# Distribution
axes[0, 1].hist(inventory_df["inventory_level"], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0, 1].set_title("Distribution of Inventory Levels")
axes[0, 1].set_xlabel("Inventory Level"); axes[0, 1].set_ylabel("Frequency")

# Day-of-week pattern
dow_avg   = inventory_df.groupby("day_of_week")["inventory_level"].mean()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
axes[1, 0].bar(day_names, dow_avg.values, color="coral", edgecolor="white")
axes[1, 0].set_title("Avg Inventory by Day of Week")
axes[1, 0].set_xlabel("Day"); axes[1, 0].set_ylabel("Avg Inventory Level")

# Monthly seasonality
monthly_avg = inventory_df.groupby("month")["inventory_level"].mean()
axes[1, 1].plot(monthly_avg.index, monthly_avg.values, marker="o", color="seagreen", linewidth=1.8)
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                             "Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=8)
axes[1, 1].set_title("Monthly Average Inventory Level")
axes[1, 1].set_xlabel("Month"); axes[1, 1].set_ylabel("Avg Inventory Level")

plt.tight_layout()
plt.show()

print("\nSummary statistics:")
print(inventory_df["inventory_level"].describe().round(2))

In [0]:
# AutoML requires a Spark DataFrame (or a registered Delta table)
spark_df = spark.createDataFrame(
    inventory_df[["date", "product_id", "inventory_level"]]
)

spark_df.printSchema()
print(f"Total rows : {spark_df.count():,}")
display(spark_df.limit(10))

In [0]:
from databricks import automl
import mlflow

mlflow.set_registry_uri("databricks-uc")

print("=" * 62)
print(" Databricks AutoML — Time Series Forecasting")
print("=" * 62)
print(f"  Target col    : inventory_level")
print(f"  Time col      : date")
print(f"  Identity cols : product_id  (one series per SKU)")
print(f"  Frequency     : daily")
print(f"  Horizon       : 30 days")
print(f"  Timeout       : 10 minutes")
print(f"  Output DB     : {CATALOG}.{SCHEMA}")
print("=" * 62 + "\n")

automl_summary = automl.forecast(
    dataset=spark_df,
    target_col="inventory_level",
    time_col="date",
    identity_col=["product_id"],
    frequency="d",
    horizon=30,
    timeout_minutes=10,
    output_database=f"{CATALOG}.{SCHEMA}",
    experiment_dir=EXPERIMENT_DIR,
)

best = automl_summary.best_trial
print("\nAutoML complete!")
print(f"  Experiment        : {automl_summary.experiment.name}")
print(f"  Best run ID       : {best.mlflow_run_id}")
print(f"  Best metrics      : {best.metrics}")
print(f"  Explore notebook  : {best.notebook_url}")

In [0]:
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from datetime import timedelta

best_run_id = automl_summary.best_trial.mlflow_run_id
model_uri   = f"runs:/{best_run_id}/model"
print(f"Loading model: {model_uri}\n")

loaded_model = mlflow.pyfunc.load_model(model_uri)

# Build future DataFrame — next 30 days x all 5 products
last_date      = inventory_df["date"].max()
future_records = [
    {"date": pd.Timestamp(d), "product_id": pid}
    for pid in inventory_df["product_id"].unique()
    for d in pd.date_range(last_date + timedelta(days=1), periods=30, freq="D")
]
future_df = pd.DataFrame(future_records)

print(f"Running inference for {len(future_df)} records ({inventory_df['product_id'].nunique()} SKUs x 30 days) ...")
predictions_df = loaded_model.predict(future_df)

print(f"Prediction columns : {predictions_df.columns.tolist()}")
display(predictions_df.head(15))

# ── Forecast visualisation ───────────────────────────────────────────────
try:
    fig, ax = plt.subplots(figsize=(14, 5))
    for idx, pid in enumerate(inventory_df["product_id"].unique()):
        clr  = plt.cm.tab10.colors[idx % 10]
        hist = inventory_df[inventory_df["product_id"] == pid].tail(90).sort_values("date")
        ax.plot(hist["date"], hist["inventory_level"],
                color=clr, lw=0.9, alpha=0.6, label=f"{pid} historical")

        preds = (
            predictions_df[predictions_df["product_id"] == pid]
            if "product_id" in predictions_df.columns
            else predictions_df
        )
        t_col = next((col for col in ["ds", "date"] if col in preds.columns), None)
        v_col = next((col for col in ["yhat", "forecast", "inventory_level"]
                      if col in preds.columns), None)
        if t_col and v_col and not preds.empty:
            ax.plot(preds[t_col], preds[v_col], "--",
                    color=clr, lw=1.3, label=f"{pid} forecast")

    ax.set_title("30-Day Inventory Forecast (dashed) vs Historical (solid)")
    ax.set_xlabel("Date"); ax.set_ylabel("Inventory Level")
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Forecast plot unavailable ({e}). Raw predictions above.")

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

print(f"Registering model: {MODEL_NAME}")
registered_mv = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name=MODEL_NAME,
)

print(f"\nModel registered successfully!")
print(f"  Name    : {registered_mv.name}")
print(f"  Version : {registered_mv.version}")
print(f"  Status  : {registered_mv.status}")

# Tag this version as the production champion
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=registered_mv.version,
)
print(f"  Alias 'champion' → version {registered_mv.version}")

# Add a descriptive tag
client.set_model_version_tag(
    name=MODEL_NAME,
    version=registered_mv.version,
    key="trained_by",
    value="databricks_automl",
)
print("  Tag 'trained_by=databricks_automl' applied.")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
    ServedModelInputWorkloadSize,
)

w             = WorkspaceClient()
model_version = str(registered_mv.version)

served_models = [
    ServedModelInput(
        model_name=MODEL_NAME,
        model_version=model_version,
        workload_size=ServedModelInputWorkloadSize.SMALL,
        scale_to_zero_enabled=True,
    )
]

try:
    # Update config if endpoint already exists
    w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Updating existing endpoint '{ENDPOINT_NAME}' to version {model_version} ...")
    endpoint = w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_models=served_models,
    )
except Exception:
    # Create brand-new endpoint
    print(f"Creating endpoint '{ENDPOINT_NAME}' (model v{model_version}) ...")
    print("Note: endpoint creation typically takes 10-20 minutes. Please wait ...")
    endpoint = w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_models=served_models),
    )

print(f"\nEndpoint ready!")
print(f"  Name  : {endpoint.name}")
print(f"  State : {endpoint.state}")
print(f"  URL   : {w.config.host}/serving-endpoints/{ENDPOINT_NAME}/invocations")

In [0]:
import requests
import json

host  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Forecast dates immediately after the training window (2024-07-01 onwards)
sample_input = {
    "dataframe_records": [
        {"date": "2024-07-01", "product_id": "SKU-001"},
        {"date": "2024-07-02", "product_id": "SKU-001"},
        {"date": "2024-07-03", "product_id": "SKU-001"},
        {"date": "2024-07-01", "product_id": "SKU-002"},
        {"date": "2024-07-02", "product_id": "SKU-002"},
    ]
}

url = f"{host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
print(f"POST {url}\n")

response = requests.post(
    url=url,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type":  "application/json",
    },
    json=sample_input,
    timeout=60,
)

print(f"Status : {response.status_code}")
if response.ok:
    print("Predictions:")
    print(json.dumps(response.json(), indent=2))
else:
    print(f"Error  : {response.text}")